# 03 — GPT-2 vs RWKV Pareto Analysis

**Pipeline:** Evaluate GPT-2 and RWKV on wiki JSONL; plot Pareto frontier (perplexity × latency)

```
Wiki JSONL shards
        │
        ├──── GPT-2 (124M)  ──┐
        └──── RWKV-4 (169M) ──┤
                              ▼
                    Evaluation metrics:
                    - perplexity
                    - tokens/sec (latency)
                    - peak GPU memory
                    - FLOPs per token
                              │
                              ▼
                    Pareto scatter plot
                    (perplexity × latency)
```

**References:**
- RWKV: Peng et al. (2023) https://arxiv.org/abs/2305.13048
- Scaling Laws: Kaplan et al. (2020) https://arxiv.org/abs/2001.08361
- Chinchilla: Hoffmann et al. (2022) https://arxiv.org/abs/2203.15556

> **CPU-friendly (small models):** 124M GPT-2 and 169M RWKV run on CPU, though slowly.

In [ ]:
from __future__ import annotations

import json
import logging
import math
import os
import time
from pathlib import Path

import torch
import wandb

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CFG = {
    "models": ["gpt2", "RWKV/rwkv-4-169m-pile"],
    "dataset": "../sync/datasets/wiki/",
    "max_tokens": 10_000,
    "stride": 512,
    "device": DEVICE,
}

if not os.getenv("WANDB_DISABLED"):
    wandb.init(project="ai-transformer-research-hub", config=CFG, job_type="pareto-eval")


In [ ]:
# Model loading, dataset ingestion, and perplexity evaluation
# RWKV: Peng et al. (2023) https://arxiv.org/abs/2305.13048
# Scaling Laws: Kaplan et al. (2020) https://arxiv.org/abs/2001.08361


def load_wiki_text(dataset_dir: Path, max_tokens: int) -> str:
    """Load text from JSONL shards up to max_tokens whitespace tokens.

    Falls back to a synthetic passage in offline/CI environments.

    Args:
        dataset_dir: Directory containing '*.jsonl' shard files.
        max_tokens: Approximate token budget.

    Returns:
        Single concatenated string of article text.
    """
    shards = sorted(dataset_dir.glob("*.jsonl"))
    if not shards:
        log.warning(
            "No JSONL shards found in %s — using synthetic text for dry-run",
            dataset_dir,
        )
        return "The Transformer architecture uses self-attention mechanisms. " * 200

    tokens_collected: list[str] = []
    for shard in shards:
        with shard.open(encoding="utf-8") as fh:
            for line in fh:
                obj = json.loads(line)
                tokens_collected.extend(obj.get("text", "").split())
                if len(tokens_collected) >= max_tokens:
                    break
        if len(tokens_collected) >= max_tokens:
            break
    return " ".join(tokens_collected[:max_tokens])


def compute_perplexity(
    model_name: str, text: str, stride: int, device: str
) -> dict:
    """Compute sliding-window perplexity for model_name on text.

    Uses the stride-based estimation from the GPT-2 paper. Supports any
    CausalLM available on HuggingFace Hub.

    Args:
        model_name: HuggingFace model identifier.
        text: Evaluation text corpus.
        stride: Tokens advanced per sliding window.
        device: Torch device string.

    Returns:
        Dict with keys: model, perplexity, latency_ms_per_token,
        peak_memory_mb, params_m.
    """
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer  # type: ignore
    except ImportError:
        log.warning("transformers not installed — returning dummy metrics for %s", model_name)
        return {
            "model": model_name, "perplexity": float("nan"),
            "latency_ms_per_token": float("nan"),
            "peak_memory_mb": 0.0, "params_m": 0.0,
        }

    log.info("Loading model: %s", model_name)
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
        model.eval().to(device)
    except Exception as exc:  # noqa: BLE001
        log.warning("Could not load %s: %s — returning dummy metrics", model_name, exc)
        return {
            "model": model_name, "perplexity": float("nan"),
            "latency_ms_per_token": float("nan"),
            "peak_memory_mb": 0.0, "params_m": 0.0,
        }

    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)
    seq_len = input_ids.size(1)
    max_len = getattr(
        model.config, "n_positions",
        getattr(model.config, "max_position_embeddings", 1024),
    )

    nlls: list[torch.Tensor] = []
    prev_end = 0
    total_time = 0.0
    num_steps = 0
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        for begin_loc in range(0, seq_len, stride):
            end_loc = min(begin_loc + max_len, seq_len)
            trg_len = end_loc - prev_end
            input_slice = input_ids[:, begin_loc:end_loc]
            target_ids = input_slice.clone()
            target_ids[:, :-trg_len] = -100

            t0 = time.perf_counter()
            outputs = model(input_slice, labels=target_ids)
            if device == "cuda":
                torch.cuda.synchronize()
            total_time += time.perf_counter() - t0
            nlls.append(outputs.loss * trg_len)
            num_steps += 1
            prev_end = end_loc
            if end_loc == seq_len:
                break

    ppl = math.exp(torch.stack(nlls).sum() / seq_len) if nlls else float("nan")
    latency_ms = (total_time / num_steps / max_len) * 1000 if num_steps else float("nan")
    peak_mem = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else 0.0

    log.info(
        "%-30s  PPL=%8.2f  lat=%.3f ms/tok  mem=%7.1f MB  params=%.0fM",
        model_name, ppl, latency_ms, peak_mem, params_m,
    )
    return {
        "model": model_name,
        "perplexity": round(ppl, 3),
        "latency_ms_per_token": round(latency_ms, 4),
        "peak_memory_mb": round(peak_mem, 1),
        "params_m": round(params_m, 1),
    }


dataset_dir = Path(CFG["dataset"])
corpus = load_wiki_text(dataset_dir, CFG["max_tokens"])
log.info("Corpus loaded: %d whitespace tokens", len(corpus.split()))

eval_results: list[dict] = []
for model_id in CFG["models"]:
    metrics = compute_perplexity(model_id, corpus, CFG["stride"], CFG["device"])
    eval_results.append(metrics)
    if not os.getenv("WANDB_DISABLED"):
        safe_name = model_id.replace("/", "_")
        wandb.log({
            f"{safe_name}/perplexity": metrics["perplexity"],
            f"{safe_name}/latency_ms_per_token": metrics["latency_ms_per_token"],
        })


In [ ]:
# Pareto frontier plot: perplexity x latency
# Chinchilla: Hoffmann et al. (2022) https://arxiv.org/abs/2203.15556
# RWKV: Peng et al. (2023) https://arxiv.org/abs/2305.13048


def is_pareto_optimal(results: list[dict]) -> list[bool]:
    """Return boolean mask of Pareto-optimal points (lower ppl AND lower latency).

    A point is on the Pareto frontier if no other point strictly dominates it
    on both objectives (perplexity and latency — both lower is better).

    Args:
        results: List of evaluation metric dicts.

    Returns:
        Boolean list, same length as results.
    """
    n = len(results)
    on_front = [True] * n
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if (
                results[j]["perplexity"] <= results[i]["perplexity"]
                and results[j]["latency_ms_per_token"] <= results[i]["latency_ms_per_token"]
                and (
                    results[j]["perplexity"] < results[i]["perplexity"]
                    or results[j]["latency_ms_per_token"] < results[i]["latency_ms_per_token"]
                )
            ):
                on_front[i] = False
                break
    return on_front


valid = [
    r for r in eval_results
    if r["perplexity"] == r["perplexity"]  # NaN check
    and r["latency_ms_per_token"] == r["latency_ms_per_token"]
]

try:
    import pandas as pd  # type: ignore
    import plotly.express as px  # type: ignore
    import plotly.graph_objects as go  # type: ignore

    if valid:
        pareto_flags = is_pareto_optimal(valid)
        df = pd.DataFrame(valid)
        df["pareto_optimal"] = pareto_flags
        df["label"] = df["model"].str.split("/").str[-1]

        fig = px.scatter(
            df,
            x="latency_ms_per_token",
            y="perplexity",
            color="label",
            size="params_m",
            symbol="pareto_optimal",
            hover_data=["params_m", "peak_memory_mb"],
            title="GPT-2 vs RWKV — Pareto Frontier (perplexity x latency)",
            labels={
                "latency_ms_per_token": "Latency (ms/token)",
                "perplexity": "Perplexity (lower is better)",
                "label": "Model",
            },
        )
        pareto_df = df[df["pareto_optimal"]].sort_values("latency_ms_per_token")
        if len(pareto_df) > 1:
            fig.add_trace(
                go.Scatter(
                    x=pareto_df["latency_ms_per_token"],
                    y=pareto_df["perplexity"],
                    mode="lines",
                    line=dict(dash="dash", color="gray"),
                    name="Pareto front",
                )
            )
        fig.show()

        if not os.getenv("WANDB_DISABLED"):
            wandb.log({"pareto_plot": wandb.Html(fig.to_html())})
    else:
        log.warning("No valid results to plot — skipping Pareto chart")

except ImportError as exc:
    log.warning("Plotting skipped — missing dependency: %s", exc)

if not os.getenv("WANDB_DISABLED"):
    wandb.finish()
